# Recompute the redundancy calibration for your own context

**If two genes look interchangeable in expression data, how often does knocking out one do what knocking out the other does?**

The published answer is pan-cancer: a ~39x lift on a 0.42% base rate, with a ceiling near 17%. Your context may differ. This notebook recomputes the whole curve from public DepMap data in a few minutes, for whatever lineage, subtype or gene set you choose.

Runs on a free Colab GPU, or on CPU more slowly. Nothing here is redistributed data: the notebook downloads the public release directly.

**What this measures, precisely.** For every gene pair: how redundant the two genes look in expression (`r_obs`, squared Pearson after lineage correction) against how similar the consequences of knocking each out are (`e_prox`, proximity on raw Chronos vectors). Then it bins by the first and reads off the second.

**What it does not measure.** Buffering. Two genes that compensate for each other perfectly appear *non*-equivalent here, precisely because each covers for the other. Single-knockout data is structurally blind to that, and this notebook inherits the blindness.

In [ ]:
#@title 1. Setup
import os, sys, urllib.request, time
import numpy as np, pandas as pd
try:
    import torch
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    !pip install -q torch
    import torch
    DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV)
if DEV == 'cpu':
    print('NOTE: on CPU this takes ~20 min rather than ~1. '
          'Runtime > Change runtime type > GPU is worth it.')

## 2. Get the data

Three public files from the DepMap portal. Set `RELEASE_URLS` to a newer release when one appears — nothing below is release-specific.

**You need to supply the download URLs.** DepMap serves files through figshare with per-release identifiers, so paste the current links for `Model.csv`, `CRISPRGeneEffect.csv` and `OmicsExpressionProteinCodingGenesTPMLogp1.csv` from [depmap.org/portal/download](https://depmap.org/portal/download/). This is deliberate: hard-coding a stale link is how a notebook silently analyses the wrong release.

In [ ]:
#@title 2. Download (paste your release URLs)
RELEASE_URLS = {
    'Model.csv': '',                                       # <- paste
    'CRISPRGeneEffect.csv': '',                            # <- paste
    'OmicsExpressionProteinCodingGenesTPMLogp1.csv': '',   # <- paste
}
os.makedirs('data', exist_ok=True)
for name, url in RELEASE_URLS.items():
    path = f'data/{name}'
    if os.path.exists(path):
        print('have', name); continue
    if not url:
        raise SystemExit(f'Paste a download URL for {name} (see cell above).')
    print('downloading', name, '...')
    urllib.request.urlretrieve(url, path)
    print('  ', round(os.path.getsize(path)/1e6), 'MB')

In [ ]:
#@title 3. Load and join
model = pd.read_csv('data/Model.csv', low_memory=False)
expr  = pd.read_csv('data/OmicsExpressionProteinCodingGenesTPMLogp1.csv', index_col=0)
chron = pd.read_csv('data/CRISPRGeneEffect.csv', index_col=0)
for df in (expr, chron):
    df.columns = [c.split(' (')[0] for c in df.columns]
expr  = expr.loc[:,  ~expr.columns.duplicated()]
chron = chron.loc[:, ~chron.columns.duplicated()]
lines = sorted(set(expr.index) & set(chron.index) & set(model.ModelID))
genes = sorted(set(expr.columns) & set(chron.columns))
expr, chron = expr.loc[lines, genes], chron.loc[lines, genes]
meta = model.set_index('ModelID').loc[lines]
print(f'{len(lines)} cell lines x {len(genes)} genes')

## 4. Choose your context

This is the cell to edit. Leave `LINEAGE = None` for the pan-cancer curve, or set it to restrict to one lineage.

**A warning that is not boilerplate.** Below a few dozen cell lines the curve degenerates: `r_obs` becomes a noisy statistic and the redundancy axis inflates wholesale. In the published 18-line subgroup, 11.3% of pairs exceeded r²=0.6, against 0.3% in a 50-line panel. The notebook warns you if your selection is small, and you should believe the warning rather than the curve.

In [ ]:
#@title 4. Context and filters
LINEAGE = None          #@param {type:"raw"}  e.g. 'Breast', 'Lung', or None
GENE_SUBSET = None      # e.g. ['BRCA1','BRCA2',...] or None for all
TAU = 0.8               #@param {type:"number"}  equivalence threshold on e_prox
EXPR_MEAN_MIN, STD_PCTL = 1.0, 20

sel = meta.index if LINEAGE is None else meta.index[meta.OncotreeLineage.fillna('') == LINEAGE]
e_np = expr.loc[sel].to_numpy(np.float64)
k_np = chron.loc[sel].to_numpy(np.float64)
k_np = np.where(np.isnan(k_np), np.nanmean(k_np, axis=0, keepdims=True), k_np)
print(f'{len(sel)} cell lines selected')
if len(sel) < 40:
    print('*** WARNING: below ~40 samples the redundancy axis inflates. ***')
    print('*** Treat any curve from this selection as uninterpretable.  ***')

expressed = e_np.mean(0) >= EXPR_MEAN_MIN
floor = np.percentile(e_np[:, expressed].std(0), STD_PCTL)
keep = expressed & (e_np.std(0) >= floor)
if GENE_SUBSET:
    keep &= np.isin(np.array(genes), GENE_SUBSET)
print(f'{int(keep.sum())} genes pass filters -> {int(keep.sum())*(int(keep.sum())-1)//2:,} pairs')

In [ ]:
#@title 5. Lineage correction (skipped automatically within one lineage)
def residualise(mat, dummies):
    X = torch.as_tensor(dummies, dtype=torch.float32, device=DEV)
    Y = torch.as_tensor(mat, dtype=torch.float32, device=DEV)
    return (Y - X @ torch.linalg.lstsq(X, Y).solution).cpu().numpy()

def zscore(m):
    return ((m - m.mean(0, keepdims=True)) / (m.std(0, keepdims=True) + 1e-12)).astype(np.float32)

if LINEAGE is None:
    lin = meta.loc[sel].OncotreeLineage.fillna('other')
    lin = lin.where(lin.map(lin.value_counts()) >= 8, 'other')
    D = pd.get_dummies(lin).to_numpy(np.float32)
    D = np.hstack([np.ones((len(lin),1), np.float32), D])
    ez = zscore(residualise(e_np, D))
    print('lineage regressed out of the expression matrix')
else:
    ez = zscore(e_np - e_np.mean(0, keepdims=True))
    print('single lineage: centred only (lineage correction is meaningless within one)')

In [ ]:
#@title 6. Compute the calibration
R_EDGES = np.linspace(0, 1, 21)
E_EDGES = np.linspace(-1.025, 1.025, 42)
CHUNK = 2048

def calibrate(ez, k_raw, keep):
    idx = np.where(keep)[0]
    E = torch.as_tensor(ez[:, idx], device=DEV)
    K = torch.as_tensor(k_raw[:, idx], dtype=torch.float32, device=DEV)
    n, G = E.shape
    norms = (K*K).sum(0)
    re_, ee_ = torch.as_tensor(R_EDGES, device=DEV), torch.as_tensor(E_EDGES, device=DEV)
    # float64 accumulator: the lowest bin can exceed float32's exact-integer range
    hist = torch.zeros((len(R_EDGES)-1)*(len(E_EDGES)-1), device=DEV, dtype=torch.float64)
    for s in range(0, G, CHUNK):
        blk = E[:, s:s+CHUNK]
        r = (blk.T @ E) / (n-1)
        dot = K[:, s:s+CHUNK].T @ K
        prox = 2*dot / (norms[s:s+CHUNK, None] + norms[None, :] + 1e-12)
        a = torch.bucketize((r**2).clamp(0, 0.999999), re_) - 1
        b = torch.bucketize(prox.clamp(-1.024, 1.024), ee_) - 1
        flat = a.clamp(0, len(R_EDGES)-2)*(len(E_EDGES)-1) + b.clamp(0, len(E_EDGES)-2)
        w = torch.ones_like(prox)
        c = blk.shape[1]
        w[torch.arange(c, device=DEV), torch.arange(s, s+c, device=DEV)] = 0
        hist.scatter_add_(0, flat.reshape(-1), w.reshape(-1).double())
    return hist.reshape(len(R_EDGES)-1, len(E_EDGES)-1).cpu().numpy() / 2

t0 = time.time()
h = calibrate(ez, k_np, keep)
print(f'done in {time.time()-t0:.0f}s')

ctr = (E_EDGES[:-1] + E_EDGES[1:]) / 2
curve = pd.DataFrame([{'r_lo': R_EDGES[i], 'pairs': h[i].sum(),
                       'p_equiv': h[i][ctr > TAU].sum()/h[i].sum() if h[i].sum() else np.nan}
                      for i in range(h.shape[0])])
base = (curve.pairs*curve.p_equiv).sum()/curve.pairs.sum()
top  = curve[(curve.r_lo >= 0.60-1e-9) & (curve.r_lo < 0.70-1e-9)]
ceil = (top.pairs*top.p_equiv).sum()/top.pairs.sum() if top.pairs.sum() else np.nan
print(f'\nbase rate      {base*100:.3f}%')
print(f'ceiling (r2 .6-.7) {ceil*100:.1f}%   over {int(top.pairs.sum()):,} pairs')
print(f'lift           {ceil/base:.1f}x')

In [ ]:
#@title 7. Plot
import matplotlib.pyplot as plt
d = curve[curve.pairs > 0]
fig, ax = plt.subplots(figsize=(8, 4.4), dpi=120)
ax.plot(d.r_lo+0.025, d.p_equiv*100, 'o-', color='#2a78d6', lw=2, ms=5)
ax.axhline(base*100, color='#52514e', ls='--', lw=1)
ax.annotate(f'base rate {base*100:.2f}%', (0.02, base*100), xytext=(0.02, base*100+1.2),
            fontsize=9, color='#52514e')
ax.set_xlabel('expression redundancy  ($r^2$, binned)')
ax.set_ylabel('% of pairs interventionally equivalent')
ax.set_title(f'{LINEAGE or "Pan-cancer"}  ({len(sel)} lines, tau={TAU})', loc='left')
for s in ('top','right'): ax.spines[s].set_visible(False)
ax.grid(axis='y', color='#e6e5e1'); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()

print('\nSparse bins (fewer than 100 pairs) are not interpretable on their own:')
print(d[d.pairs < 100][['r_lo','pairs','p_equiv']].to_string(index=False))

## 8. How to read your curve

- **The ceiling is the number to quote**, not the lift. The lift depends on which gene universe you take as the reference: in the published analysis it moved from 41x to 12x across gene strata while the ceiling stayed at 17–23%.
- **Always quote the threshold with the ceiling.** At `TAU=0.5` the published ceiling is 38%; at `TAU=0.9` it is 7%. The qualitative conclusion survives the whole range — most redundant-looking pairs are not equivalent — but the number does not.
- **Check the sparse-bin table.** A high `p_equiv` computed from twelve pairs is noise.
- **If you restricted to one lineage**, you removed the correction that matters most; expect a different and probably flatter curve.

Method, controls and the full experiment ledger: [doi:10.5281/zenodo.21988145](https://doi.org/10.5281/zenodo.21988145)